### Project Overview: Weather AI Agent Workflow 

In this project, you will build a Weather AI Agent using LLM-based agents that work together to provide weather information. The system uses a multi-agent design with memory to deliver smarter, more contextual responses.

1. **Weather Retrieval Phase (Agent 1):**
The user’s weather-related question is sent to a dedicated weather retrieval agent. This agent’s responsibility is to fetch real-time weather data (such as temperature and conditions) from an external weather source.

2. **Memory Phase (Agent 2):**
The retrieved weather information is passed to a memory agent. This agent’s role is to store and recall past weather queries and responses, enabling the system to maintain context across interactions.

3. **Response Generation Phase (Agent 3 – LLM):**
The weather data, along with any relevant memory context, is sent to an LLM-based response agent. This agent’s role is to generate a clear, natural, and user-friendly weather response.

#### Weather Agent with Memory — Diagram-Style Workflow (LLM-Based, Multi-Agent System)
```
┌──────────────┐
│   User       │
│ (Weather Q)  │
└──────┬───────┘
       │
       ▼
┌───────────────────────┐
│ Weather Retrieval     │
│ Agent (Agent 1)       │
│ - Calls Weather API   │
│ - Fetches live data   │
└──────┬────────────────┘
       │
       ▼
┌───────────────────────┐
│ Memory Agent          │
│ (Agent 2)             │
│ - Stores past queries │
│ - Recalls context     │
└──────┬────────────────┘
       │
       ▼
┌──────────────────────────┐
│ LLM Response Agent       │
│ (Agent 3)                │
│ - Combines weather data  │
│ - Uses memory context    │
│ - Generates natural reply│
└──────┬───────────────────┘
       │
       ▼
┌──────────────┐
│   User       │
│ (Final Reply)│
└──────────────┘
```


In [9]:
WEATHER_API_KEY="7dec9b347f87681d2c84520e09b594d8"

In [1]:
!pip install requests python-dotenv


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [ ]:
# Weather retrieval agent
import requests

class WeatherRetrievalAgent:
    def __init__(self, api_key):
        self.api_key = api_key
        self.base_url = "http://api.openweathermap.org/data/2.5/weather"

    def get_weather(self, city):
        params = {"q": city, "appid": self.api_key, "units": "metric"}
        res = requests.get(self.base_url, params=params)
        if res.status_code == 200:
            data = res.json()
            return {
                "city": city,
                "temperature": data["main"]["temp"],
                "humidity": data["main"]["humidity"],
                "condition": data["weather"][0]["description"]
            }
        return {"error": "City not found"}


In [3]:
# Memory Agent
class MemoryAgent:
    def __init__(self):
        self.memory = []

    def store(self, city, weather_info):
        self.memory.append({"city": city, "weather": weather_info})

    def recall(self):
        return self.memory

In [4]:
# Response Agent
class ResponseAgent:
    def generate_response(self, city, weather_info):
        if "error" in weather_info:
            return weather_info["error"]
        return (f"The current weather in {city} is {weather_info['condition']} with "
                f"a temperature of {weather_info['temperature']}°C and humidity of "
                f"{weather_info['humidity']}%.")


In [10]:
# Initialize agents
weather_agent = WeatherRetrievalAgent(WEATHER_API_KEY)
memory_agent = MemoryAgent()
response_agent = ResponseAgent()

# User input
city = input("Enter city name: ")
weather_info = weather_agent.get_weather(city)
memory_agent.store(city, weather_info)
response = response_agent.generate_response(city, weather_info)

print(response)
print("Memory:", memory_agent.recall())


City not found
Memory: [{'city': 'paris', 'weather': {'error': 'City not found'}}]


In [11]:
class WeatherAgent:
    # Initialize the WeatherAgent with the required API key
    def __init__(self, api_key):
        self.api_key = api_key  # API key for authenticating with OpenWeatherMap
        self.url = "https://api.openweathermap.org/data/2.5/weather"  # Base URL for weather API
        self.memory = Memory()  # Memory object to store past interactions or context
       # Main method to handle user queries
    def answer(self, query):
        # Extract the city name from the user's query
        city = self.extract_city(query)
        # If no city is found in the query, ask the user to specify one
        if not city:
            return "Please specify a city for weather information."
        # Fetch and return the weather information for the extracted city
        return self.get_weather(city)
 

In [12]:
def get_weather(self, city):
    # Define query parameters for the API request
    params = {
        "q": city,                 # City name for which weather data is requested
        "appid": self.api_key,     # API key for authentication
        "units": "metric"          # Return temperature in Celsius
    }
    try:
        # Send a GET request to the OpenWeatherMap API with the given parameters
        response = requests.get(self.url, params=params)
        # Raise an exception if the HTTP request returned an error status
        response.raise_for_status()
        # Parse the JSON response into a Python dictionary
        data = response.json()
        # Format and return the weather information in a user-friendly way
        return self.format_weather_response(city, data)
    except requests.exceptions.RequestException as e:
        # Handle network errors, invalid responses, or request failures
        return f"Error fetching weather data: {str(e)}"
 

In [13]:
def extract_city(self, query):
    # Convert the user query to lowercase for case-insensitive matching
    query_lower = query.lower()
    # Define common regex patterns to extract city names from user queries
    patterns = [
        r'weather in (\w+)',        # e.g., "weather in London"
        r'temperature in (\w+)',    # e.g., "temperature in Paris"
        r'forecast for (\w+)',      # e.g., "forecast for Tokyo"
        r'how is (\w+)',            # e.g., "how is Mumbai"
    ]
    # Iterate through each pattern and attempt to find a match in the query
    for pattern in patterns:
        match = re.search(pattern, query_lower)
        if match:
            # Return the extracted city name with the first letter capitalized
            return match.group(1).capitalize()
    # Return None if no city name is found in the query
    return None

In [14]:
def format_weather_response(self, city, data):
    # Recall previous weather data
    previous = self.memory.recall(city)
    current_temp = data["main"]["temp"]
    description = data["weather"][0]["description"]
    # Store current data
    self.memory.store(city, {
        "temp": current_temp,
        "description": description,
        "timestamp": time.time()
    })
    response = (
        f"The current weather in {city} is {description} "
        f"with a temperature of {current_temp}°C."
    )
    if previous:
        temp_change = current_temp - previous["temp"]
        response += f" (Temperature change: {temp_change:+.1f}°C)"
    return response

In [15]:
class Memory:
    # Initialize the memory storage as an empty dictionary
    def __init__(self):
        self.storage = {}  # Dictionary to store key–value pairs
    # Store a value in memory using a specified key
    def store(self, key, value):
        self.storage[key] = value
    # Retrieve a value from memory using the key
    # Returns None if the key does not exist
    def recall(self, key):
        return self.storage.get(key, None)
    # Clear memory contents
    # If a key is provided, remove only that entry
    # If no key is provided, clear all stored data
    def clear(self, key=None):
        if key:
            self.storage.pop(key, None)
        else:
            self.storage.clear()
 